# packages

In [2]:
#libraries and dirrectory 
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy.optimize import curve_fit
# import patientFunctions as ptfn
import seaborn as sns
import matplotlib.pyplot as plt
import re
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import plotly.express as px

# directory

In [3]:
#updates to directory management
machine_directory = 'C:/Users/mcremer' #the C and path to the project folder
# machine_directory = 'C:/Users/maega' #when working from home machine
storage_directory = 'UFL Dropbox/Maegan Cremer/research-share/Maegan/Projects' #Local, HPG, or dropbox
project_directory = 'Cardiac-Amyloidosis-Multiple-Myeloma' #project folder
project_lv2_directory = '019_1_Mistic_ClinicalSplit_CodeOcean' #deeper part of project folder

path = os.path.join(machine_directory, storage_directory, 
                    project_directory, project_lv2_directory)

# parent_dir = path
outputDir = path
os.chdir(outputDir)

# importing files

In [4]:
os.getcwd()

'C:\\Users\\mcremer\\UFL Dropbox\\Maegan Cremer\\research-share\\Maegan\\Projects\\Cardiac-Amyloidosis-Multiple-Myeloma\\019_1_Mistic_ClinicalSplit_CodeOcean'

In [5]:
#get path to this file
IGtable = pd.read_excel("CodeOcean Outputs\DescFits_clinSplit_Second_250820_v1\DescFits_clinSplit_Second__250820_v1.xlsx", sheet_name= "IG table")
Xvalues = pd.read_excel("CodeOcean Outputs/DescFits_clinSplit_Second_250820_v1/DescFits_clinSplit_Second__250820_v1.xlsx", sheet_name='X values')
feature_performance = pd.read_excel("CodeOcean Outputs/DescFits_clinSplit_Second_250820_v1/DescFits_clinSplit_Second__250820_v1.xlsx", sheet_name='06_features_performance')

inputData = pd.read_excel("DFsForSVM__202507_v1/DF_SVM_DescFits_sk__202507_v1.xlsx", sheet_name= "knownPts")


<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\mcremer\AppData\Local\Temp\ipykernel_3596\155526584.py:2: SyntaxWarning: invalid escape sequence '\D'
  IGtable = pd.read_excel("CodeOcean Outputs\DescFits_clinSplit_Second_250820_v1\DescFits_clinSplit_Second__250820_v1.xlsx", sheet_name= "IG table")


In [6]:
inputData.index = inputData['DeID']

In [7]:
y_all = inputData['CA_status_yes']

# SHAP plots

In [8]:
feature_rank = feature_performance.T.iloc[0]

In [9]:

nbins = 10
scale = 0.1
n_features = 4
spacing = 2
min_max_IG = 2.5

fig = go.Figure()
fig.update_layout(
    plot_bgcolor='white',
    autosize=False,
    width=1000,
    height=600,
    coloraxis = {'colorscale':'Bluered'},
    xaxis_title = "Integrated Gradient",
)
 
fig.update_xaxes(
    zeroline=True,
    zerolinecolor="black",
    range = [-min_max_IG,min_max_IG]
    )
fig.update_yaxes(
    zeroline=True,
    zerolinecolor="black",
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey',
    range = [-spacing, spacing*n_features + spacing]
)
 
# feature_rank = np.argsort(np.sum(abs(IGtable),axis=0))
feature_order = feature_rank[:n_features][::-1]

y_val = 1
bin_ids = list(range(nbins))
for f in feature_order:
    bins = np.linspace(IGtable.loc[:,f].min()*1.1,IGtable.loc[:,f].max()*1.1,nbins+1)
    ig_bins = pd.cut(IGtable.loc[:,f], bins=bins, labels=bin_ids)
 
    x_vals = []
    y_vals = []
    f_vals = []
    for bin in bin_ids:
        x_bin_vals = list(IGtable.loc[ig_bins[ig_bins == bin].index,f].values)
        n_pts = len(x_bin_vals)
        if n_pts > 0:
            x_vals = x_vals + x_bin_vals
            y_vals = y_vals + list(y_val + scale*(np.array(list(range(n_pts)))-(n_pts-1)/2))
            f_vals = f_vals + list(Xvalues.loc[ig_bins[ig_bins == bin].index,f].values)
    fig.add_trace(go.Scatter(
        x = x_vals, 
        y = y_vals,
        mode = 'markers',
        marker = dict(size=12,
                      color = f_vals,
                      coloraxis = "coloraxis",
                     ),
        name=f,
    ))
    y_val += spacing
 
 
fig.update_layout(showlegend=False, coloraxis_showscale=True,font=dict(size=20))
fig.update_coloraxes(colorbar_showticklabels=False,
                    colorbar_title=dict(text="Feature Value",side = "right"),
                    cmin = -1, cmax = 1)
fig.update_yaxes(tickvals=[1]+[1 + spacing*i for i in range(1,n_features)], 
                 ticktext=feature_order)
 
fig.show()

In [25]:

nbins = 10
scale = 0.1
n_features = 4
spacing = 2
min_max_IG = 2.5

fig = go.Figure()
fig.update_layout(
    plot_bgcolor='white',
    autosize=False,
    width=1000,
    height=600,
    coloraxis = {'colorscale':'Bluered'},
    xaxis_title = "Integrated Gradient",
)
 
fig.update_xaxes(
    zeroline=True,
    zerolinecolor="black",
    range = [-min_max_IG,min_max_IG]
    )
fig.update_yaxes(
    zeroline=True,
    zerolinecolor="black",
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey',
    range = [-spacing, spacing*n_features + spacing]
)
 
# feature_rank = np.argsort(np.sum(abs(IGtable),axis=0))
feature_order = feature_rank[:n_features][::-1]

y_val = 1
bin_ids = list(range(nbins))
for f in feature_order:
    bins = np.linspace(IGtable.loc[:,f].min()*1.1,IGtable.loc[:,f].max()*1.1,nbins+1)
    ig_bins = pd.cut(IGtable.loc[:,f], bins=bins, labels=bin_ids)
 
    x_vals = []
    y_vals = []
    f_vals = []
    id_vals = []
    for bin in bin_ids:
        x_bin_vals = list(IGtable.loc[ig_bins[ig_bins == bin].index,f].values)
        n_pts = len(x_bin_vals)
        if n_pts > 0:
            x_vals = x_vals + x_bin_vals
            y_vals = y_vals + list(y_val + scale*(np.array(list(range(n_pts)))-(n_pts-1)/2))
            f_vals = f_vals + list(Xvalues.loc[ig_bins[ig_bins == bin].index,f].values)
            id_vals = id_vals + list(IGtable.loc[ig_bins[ig_bins == bin].index,f].index)
    fig.add_trace(go.Scatter(
        x = x_vals, 
        y = y_vals,
        mode = 'markers',
        marker = dict(size=16,
                      color = f_vals,
                      coloraxis = "coloraxis",
                      #marker symbol for F-03 and Y-02 as stars
                      symbol = ['star' if id in ['Y-02', 'F-01'] else 'circle' for id in id_vals],
                      line_color = 'white', line_width= 1,
                     ),
        name=f,
        text = id_vals, textposition= 'bottom center'
    ))
    y_val += spacing
 
 
fig.update_layout(showlegend=False, coloraxis_showscale=True,font=dict(size=20))
fig.update_coloraxes(colorbar_showticklabels=False,
                    colorbar_title=dict(text="Feature Value",side = "right"),
                    cmin = -1, cmax = 1)
fig.update_yaxes(tickvals=[1]+[1 + spacing*i for i in range(1,n_features)], 
                 ticktext=feature_order)
 
fig.show()

In [24]:
testSetPatients = ['T-01', 'B-02', 'I-01', 'Y-02', 'F-03', 'B-03', 'F-01', 'G-03', 'Q-01', 'C-01']

IGtable_filtered = IGtable[IGtable['DeID'].isin(testSetPatients)]
Xvalues_filtered = Xvalues[Xvalues['IG'].isin(testSetPatients)]

nbins = 10
scale = 0.1
n_features = 4
spacing = 2
min_max_IG = 2.5

fig = go.Figure()
fig.update_layout(
    plot_bgcolor='white',
    autosize=False,
    width=1000,
    height=600,
    coloraxis = {'colorscale':'Bluered'},
    xaxis_title = "Integrated Gradient",
)
 
fig.update_xaxes(
    zeroline=True,
    zerolinecolor="black",
    range = [-min_max_IG,min_max_IG]
    )
fig.update_yaxes(
    zeroline=True,
    zerolinecolor="black",
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey',
    range = [-spacing, spacing*n_features + spacing]
)
 
# feature_rank = np.argsort(np.sum(abs(IGtable),axis=0))
feature_order = feature_rank[:n_features][::-1]

y_val = 1
bin_ids = list(range(nbins))
for f in feature_order:
    bins = np.linspace(IGtable_filtered.loc[:,f].min()*1.1,IGtable_filtered.loc[:,f].max()*1.1,nbins+1)
    ig_bins = pd.cut(IGtable_filtered.loc[:,f], bins=bins, labels=bin_ids)
 
    x_vals = []
    y_vals = []
    f_vals = []
    id_vals = []
    for bin in bin_ids:
        x_bin_vals = list(IGtable_filtered.loc[ig_bins[ig_bins == bin].index,f].values)
        n_pts = len(x_bin_vals)
        if n_pts > 0:
            x_vals = x_vals + x_bin_vals
            y_vals = y_vals + list(y_val + scale*(np.array(list(range(n_pts)))-(n_pts-1)/2))
            f_vals = f_vals + list(Xvalues_filtered.loc[ig_bins[ig_bins == bin].index,f].values)
            id_vals = id_vals + list(IGtable_filtered.loc[ig_bins[ig_bins == bin].index,f].index)
    fig.add_trace(go.Scatter(
        x = x_vals, 
        y = y_vals,
        mode = 'markers',
        marker = dict(size=16,
                      color = f_vals,
                      coloraxis = "coloraxis",
                      #marker symbol for F-03 and Y-02 as stars (need to convert to list first
                        symbol = ['star' if id in ['Y-02', 'F-01'] else 'circle' for id in id_vals],
                        line_color = 'white', line_width= 1,
                     ),
        name=f,
        text = id_vals, textposition= 'bottom center'
    ))
    y_val += spacing
 
 
fig.update_layout(showlegend=False, coloraxis_showscale=True,font=dict(size=20))
fig.update_coloraxes(colorbar_showticklabels=False,
                    colorbar_title=dict(text="Feature Value",side = "right"),
                    cmin = -1, cmax = 1)
fig.update_yaxes(tickvals=[1]+[1 + spacing*i for i in range(1,n_features)], 
                 ticktext=feature_order)
 
fig.show()

In [12]:
ig_bins

0     2
4     9
6     2
15    3
20    5
22    0
23    1
26    0
27    2
35    0
Name: Chloride_linear_intercept, dtype: category
Categories (10, int64): [0 < 1 < 2 < 3 ... 6 < 7 < 8 < 9]

In [13]:
list(IGtable_filtered.loc[ig_bins[ig_bins == bin].index,f].index)

[4]

In [14]:
IGtable_filtered.loc[ig_bins[ig_bins == bin].index,f]

4    3.130658
Name: Chloride_linear_intercept, dtype: float64

In [15]:
IGtable_filtered

,DeID,BUN_50%,Chloride_linear_intercept,NT Pro BNP_exp_intercept,Serum Electrophoresis Beta_linear_intercept
0,B-02,-0.511194,-0.640408,-0.032282,-0.151765
4,F-03,2.499720,3.130658,0.158030,0.736773
6,I-01,-0.531298,-0.665417,-0.033582,-0.157466
15,T-01,-0.084389,-0.106135,-0.005256,-0.025804
20,Y-02,0.830978,1.040708,0.052540,0.243443
22,B-03,-1.050144,-1.316598,-0.066154,-0.309134
23,C-01,-0.935841,-1.171265,-0.059301,-0.274553
26,F-01,-1.271688,-1.592585,-0.080404,-0.376108
27,G-03,-0.558873,-0.700041,-0.035309,-0.165963
35,Q-01,-1.179909,-1.476452,-0.074812,-0.346649


# IG plots

In [16]:
IGtable_labeled= IGtable
IGtable_labeled.index= IGtable['DeID']
IG_X_all_labeled= Xvalues
IG_X_all_labeled.index = Xvalues['IG']

In [17]:
#marker dimensions
IGtable_labeled['Shape'] = ['diamond' if status == 1 else 'circle' for status in y_all]
IGtable_labeled['size'] = [2 for i in range(len(y_all))]

In [18]:
pt_interest = ['F-01', "Y-02"]
IGtable_labeled.loc['F-01','Shape'] = 'star'
IGtable_labeled.loc['Y-02','Shape'] = 'star'
IGtable_labeled.loc['F-01','size'] = 6
IGtable_labeled.loc['Y-02','size'] = 6

In [19]:
#function for moving the text around on the integrated gradient plot to improve the visualization
def text_pos(df, valueCol):
    df.sort_values(by = valueCol, ascending = False) #sorting the values first
    positions = ['middle right', 'middle left'] #locations for text
    return [positions[i %len(positions)] for i in range(len(df[valueCol]))]

In [20]:
IGtable_labeled['label'] = [ptID if ptID in pt_interest else np.nan for ptID in IGtable_labeled['DeID']]

## only narrative cases labeled

In [21]:
feature_combinations = [[1,2], [1,3], [1,4], 
                        [2,1], [2,3], [2,4], 
                        [3,1], [3,2], [3,4],
                        [4,1], [4,2], [4,3]]

for combo in feature_combinations: 
    a, b = combo
    feat_1 = feature_order[::-1].iloc[a-1]
    feat_2 = feature_order[::-1].iloc[b-1]
    
    
    fig=px.scatter(x = IG_X_all_labeled[feat_1], y = IGtable_labeled[feat_1], 
                     text= IGtable_labeled['label'], 
                     color = IG_X_all_labeled[feat_2],
                     symbol = IGtable_labeled['Shape'], size = IGtable_labeled['size'],
                     symbol_sequence = IGtable_labeled['Shape'].unique().tolist()
                    )
    fig.update_traces(textposition = text_pos(IGtable_labeled, feat_1))



    fig.update_layout(
        plot_bgcolor='white',
        autosize=False,
        width=800,
        height=800,
        coloraxis = {'colorscale':'Bluered'},
        xaxis_title = feat_1,
        yaxis_title = "Integrated Gradient for " + feat_1,
        font=dict(size=12),
        legend = dict(orientation = 'h', yanchor = 'bottom', y= 1.02, xanchor = 'right', x = 1)
    )

    fig.update_xaxes(
        zeroline=True,
        zerolinecolor='lightgrey',
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey'
        )

    fig.update_yaxes(
        zeroline=True,
        zerolinecolor='lightgrey',
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey'
    )

    fig.update_coloraxes(colorbar_showticklabels=True,
                        colorbar_title=dict(text=feat_2,side = "right"),
                        cmin = IG_X_all_labeled[feat_2].min(), cmax = IG_X_all_labeled[feat_2].max())

    fig.show()


## only narrative cases are labled and the values are set to the feature values not the standard scalars

In [22]:
feature_combinations = [[1,2], [1,3], [1,4], 
                        [2,1], [2,3], [2,4], 
                        [3,1], [3,2], [3,4],
                        [4,1], [4,2], [4,3]]

for combo in feature_combinations: 
    a, b = combo
    feat_1 = feature_order[::-1].iloc[a-1]
    feat_2 = feature_order[::-1].iloc[b-1]
    
    
    fig=px.scatter(x = inputData[feat_1], y = IGtable_labeled[feat_1], 
                     text= IGtable_labeled['label'], 
                     color = inputData[feat_2],
                     symbol = IGtable_labeled['Shape'], size = IGtable_labeled['size'],
                     symbol_sequence = IGtable_labeled['Shape'].unique().tolist()
                    )
    fig.update_traces(textposition = text_pos(IGtable_labeled, feat_1))



    fig.update_layout(
        plot_bgcolor='white',
        autosize=False,
        width=800,
        height=800,
        coloraxis = {'colorscale':'Bluered'},
        xaxis_title = feat_1,
        yaxis_title = "Integrated Gradient for " + feat_1,
        font=dict(size=12),
        legend = dict(orientation = 'h', yanchor = 'bottom', y= 1.02, xanchor = 'right', x = 1)
    )

    fig.update_xaxes(
        zeroline=True,
        zerolinecolor='lightgrey',
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey'
        )

    fig.update_yaxes(
        zeroline=True,
        zerolinecolor='lightgrey',
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey'
    )

    fig.update_coloraxes(colorbar_showticklabels=True,
                        colorbar_title=dict(text=feat_2,side = "right"),
                        cmin = inputData[feat_2].min(), cmax = inputData[feat_2].max())

    fig.show()


## all patients are labeled

In [23]:
feature_combinations = [[1,2], [1,3], [1,4], 
                        [2,1], [2,3], [2,4], 
                        [3,1], [3,2], [3,4],
                        [4,1], [4,2], [4,3]]

for combo in feature_combinations: 
    a, b = combo
    feat_1 = feature_order.iloc[a-1]
    feat_2 = feature_order.iloc[b-1]
    
    
    fig=px.scatter(x = IG_X_all_labeled[feat_1], y = IGtable_labeled[feat_1], 
                     text= IGtable_labeled.index, 
                     color = IG_X_all_labeled[feat_2],
                     symbol = IGtable_labeled['Shape'], size = IGtable_labeled['size'],
                     symbol_sequence = IGtable_labeled['Shape'].unique().tolist()
                    )
    fig.update_traces(textposition = text_pos(IGtable_labeled, feat_1))



    fig.update_layout(
        plot_bgcolor='white',
        autosize=False,
        width=800,
        height=800,
        coloraxis = {'colorscale':'Bluered'},
        xaxis_title = feat_1,
        yaxis_title = "Integrated Gradient for " + feat_1,
        font=dict(size=12),
        legend = dict(orientation = 'h', yanchor = 'bottom', y= 1.02, xanchor = 'right', x = 1)
    )

    fig.update_xaxes(
        zeroline=True,
        zerolinecolor='lightgrey',
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey'
        )

    fig.update_yaxes(
        zeroline=True,
        zerolinecolor='lightgrey',
        mirror=True,
        ticks='outside',
        showline=True,
        linecolor='black',
        gridcolor='lightgrey'
    )

    fig.update_coloraxes(colorbar_showticklabels=True,
                        colorbar_title=dict(text=feat_2,side = "right"),
                        cmin = IG_X_all_labeled[feat_2].min(), cmax = IG_X_all_labeled[feat_2].max())

    fig.show()
